In [ ]:
from LLTB import lltb
from dataset import CallData, BootstrapSplitter
from sklearn.metrics import mean_squared_error, log_loss, zero_one_loss
import numpy as np
import pandas as pd
import copy
from mip import *
import time

In [ ]:
dataset_name = 'california_housing_price'
dataset = CallData(standardize=True).call(dataset_name = dataset_name, data_format = 'df')
X = dataset.X
y = dataset.y
columns = X.columns


sample_size = min(500,int(len(X)))
splitter = BootstrapSplitter(reps=1, train_size=sample_size, random_state=111, replace=True)
for ind, (train_ind, test_ind) in enumerate(splitter.split(X, y)):
    if ind == 0:
        x_train = X.iloc[train_ind]
        y_train = y.iloc[train_ind]
        x_test = X.iloc[test_ind]
        y_test = y.iloc[test_ind]
x_mean = np.round(dataset.x_means, decimals=10)
x_std = np.round(dataset.x_stds, decimals=10)
if dataset.learning_type == 'linreg':
    task = 'regression'
    y_mean = np.round(dataset.y_means, decimals=10)
    y_std = np.round(dataset.y_stds, decimals=10)
    loss_func = mean_squared_error
else:
    task = 'classification'
    y_mean = None
    y_std = None
    loss_func = log_loss


In [200]:
lltb_empty_model = lltb.LinRepRuleEnsemble(num_rules=0, task=dataset.learning_type, max_complexity=5, reg=10,
                                       epsilon_r=0.0, weighted_logloss=True, max_digits=2,
                                       no_refit=False, greedy=False, x_std=x_std, x_mean=x_mean).fit(x_train,y_train)
empty_model_train_loss = loss_func(y_train, lltb_empty_model.predict(x_train))
lltb_empty_model.rules(), lltb_empty_model.digit_complexity(), lltb_empty_model.model_complexity(), empty_model_train_loss

(['-0.0091'], 0, 0, 1.0652404871367647)

In [ ]:
lltb_model = lltb.LinRepRuleEnsemble(num_rules=5, task=dataset.learning_type, max_complexity=5, reg=10,
                                       epsilon_r=0.0, weighted_logloss=True, max_digits=2,
                                       no_refit=False, greedy=False, x_std=x_std, x_mean=x_mean, axis_paral=False).fit(x_train,y_train)
lltb_model.rules(rounding_digits=16), lltb_model.digit_complexity(), lltb_model.model_complexity(), loss_func(y_train, lltb_model.predict(x_train))/empty_model_train_loss

(['-0.7192907288128441',
  '0.4567724870615532 if -3.9206359119755527 Latitude -3.6741536320416794 Longitude ≥ 0.2814871137297915',
  '0.3540602334375925 if -2.8328191571374699 Latitude -2.4436193453827277 Longitude ≥ 0.0545544012363669 &  -Longitude ≥ -0.9461671433170230',
  '-0.0576994917725305 if -1.5691798838118154 Latitude -1.5276017218890570 Longitude ≥ 0.0516061653810011 &  -Latitude ≥ 0.3933131776964299',
  '0.4626271494737123 if -0.3638750062523229 Latitude -1.1563646692389744 Longitude ≥ -0.8831049175948164 &  -Latitude ≥ 0.3309578001480482 &  Longitude ≥ 0.0987402993734557 &  Longitude ≥ 0.3237460234607501',
  '0.3075817250930041 if -Longitude ≥ 0.7584384313263144 &  -Latitude ≥ -1.4562458044772544'],
 127,
 31,
 0.7295211556281562)

In [ ]:
lltb_model_copy = copy.deepcopy(lltb_model)
simple_lltb = lltb.SimplifiedRuleEnsemble(model=lltb_model_copy)
lltb_model_copy, all_permutations = simple_lltb.truncate_decimal_digits_param_by_param_extensively(x=x_train, y=y_train, risk_increase_thrsh=0.0, round=True, 
                                                                                 seed=None, number_of_permutations=None, return_all_enumerations=True)
norm_train_loss = loss_func(y_train, lltb_model_copy.predict(x_train))/empty_model_train_loss
deviation_from_original = norm_train_loss - loss_func(y_train, lltb_model.predict(x_train))/empty_model_train_loss
lltb_model_copy.rules(), lltb_model_copy.digit_complexity(), lltb_model_copy.model_complexity(), norm_train_loss, deviation_from_original

In [ ]:
simple_mip_model = copy.deepcopy(lltb_model)
# x_ref = x_train.to_numpy()
unique_indices = np.unique(train_ind)
x_ref = X.to_numpy()[unique_indices]
sig_levels = [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# sig_levels = [ 1, 2, 3, 4]
for ci, c in enumerate(lltb_model.conditions_):
    num_prop = c.w_.shape[1]
    for j in range(num_prop):
        original_a = c.w_[:,j]
        original_t = -c.t_[j]

        nonzero_original_a = original_a[np.nonzero(original_a)[0]]
        x_ref_nonzero = x_ref[:, np.nonzero(original_a)[0]]

        coeff_candidate_sets = [
            build_candidate_values(nonzero_original_a[j], sig_levels, include_original=True, use_round=True, use_trunc=False, use_integr=True)
            for j in range(len(nonzero_original_a))
        ]
        
        threshold_candidate_set = build_candidate_values(original_t, sig_levels, include_original=True, use_round=True, use_trunc=False, use_integr=False)


        if ci == 1:
            rule_2_candidate_sets = [coeff_candidate_sets, threshold_candidate_set]

        # print("Candidate sets:")
        # for j, cand in enumerate(coeff_candidate_sets):
        #     print(f"  a_{j}: {cand}")
        # print(f"  t: {threshold_candidate_set}")

        # Solve with mismatch penalty + optional hard cap
        selected_samples_by_original_prop_j = (nonzero_original_a @ x_ref_nonzero.T - original_t >= 0).sum()
        result = simplify_proposition_mip(
            X=x_ref_nonzero,
            original_a=nonzero_original_a,
            original_t=original_t,
            coeff_candidate_sets=coeff_candidate_sets,
            threshold_candidate_set=threshold_candidate_set,
            cost_fn=default_complexity_cost,
            margin=1e-16,
            max_mismatches=int(0.0 * selected_samples_by_original_prop_j),         # exact matching on reference set
            mismatch_penalty=None,    # not needed if max_mismatches=0
            verbose=False,
        )


        simplified_a = np.zeros_like(original_a)
        simplified_a[np.nonzero(original_a)[0]] = result.simplified_a
        simple_mip_model.conditions_[ci].w_[:,j] = simplified_a
        simple_mip_model.conditions_[ci].t_[j] = float(-result.simplified_t)
        print(f"number of mismatches for rule {ci}, proposition {j}: {result.mismatches}")


simple_mip_model.rules(), simple_mip_model.digit_complexity(), simple_mip_model.model_complexity(), loss_func(y_train, simple_mip_model.predict(x_train))/empty_model_train_loss

In [ ]:
def run_experiment(dataset_name, method, mismaches_threshold=[0.0]):

    dataset = CallData(standardize=True).call(dataset_name = dataset_name, data_format = 'df')
    X = dataset.X
    y = dataset.y
    y_copy_for_gb = y.copy()
    reps = 1
    num_rules = 12
    l2_lam = [1e-6, 1e-3, 1e-1, 1, 10]
    l2_lam_size = len(l2_lam)
    x_mean = np.round(dataset.x_means, decimals=5)
    x_std = np.round(dataset.x_stds, decimals=5)
    if dataset.learning_type == 'linreg':
        task = 'regression'
        y_mean = np.round(dataset.y_means, decimals=5)
        y_std = np.round(dataset.y_stds, decimals=5)
        loss_fnc = mean_squared_error
    else:
        task = 'classification'
        y_mean = None
        y_std = None
        loss_fnc = log_loss
        y_copy_for_gb[y_copy_for_gb==0] = -1
    sample_size = min(500,int(len(X)))
    splitter = BootstrapSplitter(reps=reps, train_size=sample_size, random_state=111, replace=True)
    lam_reg = 1e-2

    model_df = pd.DataFrame()
    m = 0
    mismaches_threshold_list = mismaches_threshold
    max_comp = min(5, X.shape[1])
    digits = np.arange(0,16)

    for ind, (train_ind, test_ind) in enumerate(splitter.split(X, y)):
        x_train = X.iloc[train_ind]
        x_test = X.iloc[test_ind]
        if method == 'axis':
            y_train = y_copy_for_gb.iloc[train_ind]
            y_test = y_copy_for_gb.iloc[test_ind]
        else:
            y_train = y.iloc[train_ind]
            y_test = y.iloc[test_ind]

        train_unique_indices = np.unique(train_ind)
        x_ref = X.to_numpy()[train_unique_indices]
        y_ref = y.to_numpy()[train_unique_indices]
        df_lltb = pd.read_excel("./experiments_results/3000_samples_15_rules/lltb_outcv_boot_3000_samples_"+dataset_name+".xlsx")
        lam_reg = df_lltb[(df_lltb['rep no'] == 1) & (df_lltb['max rule no'] == num_rules)]['lambda'].item()
        model = lltb.LinRepRuleEnsemble(num_rules=num_rules, task=dataset.learning_type, max_complexity=max_comp, reg=lam_reg,
                                        epsilon_r=0.0, weighted_logloss=True, no_refit=False, greedy=False,
                                        max_digits=16, axis_paral=False).fit(x_train,y_train)
        actual_num_rules = len(model.ensembles_)

        for nr in range(actual_num_rules):
            model_rule_original_copy = copy.deepcopy(model.ensembles_[nr])
            model_df.loc[m,'l2_lam'] = lam_reg
            model_df.loc[m,'rep'] = ind
            model_df.loc[m,'num_rules'] = nr+1
            model_df.loc[m,'complexity'] = model_rule_original_copy.model_complexity()
            model_df.loc[m,'num_conditions'] = model_rule_original_copy.num_of_rules()
            model_df.loc[m,'num_propositions'] = model_rule_original_copy.num_of_propositions()
            model_df.loc[m,'train_loss'] = loss_fnc(y_train, model_rule_original_copy.predict_aux(x_train)) if method == 'axis' else loss_fnc(y_train, model_rule_original_copy.predict(x_train))
            model_df.loc[m,'test_loss'] = loss_fnc(y_test, model_rule_original_copy.predict_aux(x_test)) if method == 'axis' else loss_fnc(y_test, model_rule_original_copy.predict(x_test))
            model_df.loc[m,'ref_loss'] = loss_fnc(y_ref, model_rule_original_copy.predict_aux(x_ref)) if method == 'axis' else loss_fnc(y_ref, model_rule_original_copy.predict(x_ref))
            if task == 'classification':
                model_df.loc[m,'train_01_loss'] = zero_one_loss(y_train, np.where(model_rule_original_copy.predict_aux(x_train) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_train, np.where(model_rule_original_copy.predict(x_train) > 0.5, 1, 0))
                model_df.loc[m,'test_01_loss'] = zero_one_loss(y_test, np.where(model_rule_original_copy.predict_aux(x_test) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_test, np.where(model_rule_original_copy.predict(x_test) > 0.5, 1, 0))
                model_df.loc[m,'ref_01_loss'] = zero_one_loss(y_ref, np.where(model_rule_original_copy.predict_aux(x_ref) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_ref, np.where(model_rule_original_copy.predict(x_ref) > 0.5, 1, 0))
            else:
                model_df.loc[m,'train_01_loss'] = 0
                model_df.loc[m,'test_01_loss'] = 0
                model_df.loc[m,'ref_01_loss'] = 0
            model_df.loc[m,'digits_complexity'] = model_rule_original_copy.digit_complexity()
            model_df.loc[m,'total_complexity'] = model_rule_original_copy.model_complexity() + model_rule_original_copy.digit_complexity()
            model_df.loc[m,'digits'] = 'original'
            model_df.loc[m,'simplifying_method'] = 'none'
            model_df.loc[m,'seed'] = None
            model_df.loc[m,'delta'] = None
            model_df.loc[m,'comp_time'] = None
    

            m += 1

            if method == 'axis':
                seed_list = [None]
            else:
                seed_list = [123]

            for delta in mismaches_threshold_list:
                for sd in seed_list:
                    simple_model_copy = copy.deepcopy(model.ensembles_[nr])
                    simple_model = lltb.SimplifiedRuleEnsemble(model=simple_model_copy)
                    start_time = time.time()
                    simple_model = simple_model.truncate_decimal_digits_param_by_param_extensively(x=x_ref, y=y_ref, risk_increase_thrsh=delta, round=True, seed=sd, number_of_permutations=None)
                    end_time = time.time()
                    model_df.loc[m,'comp_time'] = end_time - start_time
                    model_df.loc[m,'l2_lam'] = lam_reg
                    model_df.loc[m,'rep'] = ind
                    model_df.loc[m,'num_rules'] = nr+1
                    model_df.loc[m,'complexity'] = simple_model.model_complexity()
                    model_df.loc[m,'num_conditions'] = simple_model.num_of_rules()
                    model_df.loc[m,'num_propositions'] = simple_model.num_of_propositions()
                    model_df.loc[m,'train_loss'] = loss_fnc(y_train, simple_model.predict_aux(x_train)) if method == 'axis' else loss_fnc(y_train, simple_model.predict(x_train))
                    model_df.loc[m,'test_loss'] = loss_fnc(y_test, simple_model.predict_aux(x_test)) if method == 'axis' else loss_fnc(y_test, simple_model.predict(x_test))
                    model_df.loc[m,'ref_loss'] = loss_fnc(y_ref, simple_model.predict_aux(x_ref)) if method == 'axis' else loss_fnc(y_ref, simple_model.predict(x_ref))
                    if task == 'classification':
                        model_df.loc[m,'train_01_loss'] = zero_one_loss(y_train, np.where(simple_model.predict_aux(x_train) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_train, np.where(simple_model.predict(x_train) > 0.5, 1, 0))
                        model_df.loc[m,'test_01_loss'] = zero_one_loss(y_test, np.where(simple_model.predict_aux(x_test) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_test, np.where(simple_model.predict(x_test) > 0.5, 1, 0))
                        model_df.loc[m,'ref_01_loss'] = zero_one_loss(y_ref, np.where(simple_model.predict_aux(x_ref) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_ref, np.where(simple_model.predict(x_ref) > 0.5, 1, 0))
                    else:
                        model_df.loc[m,'train_01_loss'] = 0
                        model_df.loc[m,'test_01_loss'] = 0
                        model_df.loc[m,'ref_01_loss'] = 0
                    model_df.loc[m,'digits_complexity'] = simple_model.digit_complexity()
                    model_df.loc[m,'total_complexity'] = simple_model.model_complexity() + simple_model.digit_complexity()
                    model_df.loc[m,'digits'] = 'variable'
                    model_df.loc[m,'simplifying_method'] = 'param_by_param_rounding'
                    model_df.loc[m,'seed'] = sd
                    model_df.loc[m,'delta'] = delta

                    m += 1

            for dig in digits:
                rounded_model_copy = copy.deepcopy(model.ensembles_[nr])
                simple_rounded_model = lltb.SimplifiedRuleEnsemble(model=rounded_model_copy)
                simple_rounded_model = simple_rounded_model.truncate_decimal_digits_whole_ensemble(max_digits=dig, round=True)
                model_df.loc[m,'l2_lam'] = lam_reg
                model_df.loc[m,'rep'] = ind
                model_df.loc[m,'num_rules'] = nr+1
                model_df.loc[m,'complexity'] = simple_rounded_model.model_complexity()
                model_df.loc[m,'num_conditions'] = simple_rounded_model.num_of_rules()
                model_df.loc[m,'num_propositions'] = simple_rounded_model.num_of_propositions()
                model_df.loc[m,'train_loss'] = loss_fnc(y_train, simple_rounded_model.predict_aux(x_train)) if method == 'axis' else loss_fnc(y_train, simple_rounded_model.predict(x_train))
                model_df.loc[m,'test_loss'] = loss_fnc(y_test, simple_rounded_model.predict_aux(x_test)) if method == 'axis' else loss_fnc(y_test, simple_rounded_model.predict(x_test))
                model_df.loc[m,'ref_loss'] = loss_fnc(y_ref, simple_rounded_model.predict_aux(x_ref)) if method == 'axis' else loss_fnc(y_ref, simple_rounded_model.predict(x_ref))
                if task == 'classification':
                    model_df.loc[m,'train_01_loss'] = zero_one_loss(y_train, np.where(simple_rounded_model.predict_aux(x_train) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_train, np.where(simple_rounded_model.predict(x_train) > 0.5, 1, 0))
                    model_df.loc[m,'test_01_loss'] = zero_one_loss(y_test, np.where(simple_rounded_model.predict_aux(x_test) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_test, np.where(simple_rounded_model.predict(x_test) > 0.5, 1, 0))
                    model_df.loc[m,'ref_01_loss'] = zero_one_loss(y_ref, np.where(simple_rounded_model.predict_aux(x_ref) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_ref, np.where(simple_rounded_model.predict(x_ref) > 0.5, 1, 0))
                else:
                    model_df.loc[m,'train_01_loss'] = 0
                    model_df.loc[m,'test_01_loss'] = 0
                    model_df.loc[m,'ref_01_loss'] = 0
                model_df.loc[m,'digits_complexity'] = simple_rounded_model.digit_complexity()
                model_df.loc[m,'total_complexity'] = simple_rounded_model.model_complexity() + simple_rounded_model.digit_complexity()
                model_df.loc[m,'digits'] = dig
                model_df.loc[m,'simplifying_method'] = 'whole_ensemble_rounding'
                model_df.loc[m,'seed'] = None
                model_df.loc[m,'delta'] = None
                model_df.loc[m,'comp_time'] = None
                m += 1
                
            if method == 'oblique':
                for delta in mismaches_threshold_list:
                    simple_model_by_mip_copy = copy.deepcopy(model.ensembles_[nr])
                    if delta > 0:
                        sig_levels = [1, 2, 3]
                    else:
                        sig_levels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
                    start_time = time.time()
                    for ci, c in enumerate(simple_model_by_mip_copy.conditions_):
                        num_prop = c.w_.shape[1]
                        mismatches_identified = False
                        for j in range(num_prop):
                            original_a = c.w_[:,j]
                            nonzero_original_a = original_a[np.nonzero(original_a)[0]]
                            x_ref_nonzero = x_ref[:, np.nonzero(original_a)[0]]
                            original_t = -c.t_[j]

                            coeff_candidate_sets = [
                                build_candidate_values(nonzero_original_a[j], sig_levels, include_original=True, use_round=True, use_trunc=True, use_integr=True)
                                for j in range(len(nonzero_original_a))
                            ]
                            
                            threshold_candidate_set = build_candidate_values(original_t, sig_levels, include_original=True, use_round=True, use_trunc=True, use_integr=True)

                            
                            selected_samples_by_original_prop_j = (nonzero_original_a @ x_ref_nonzero.T - original_t >= 0).sum()
                            # Solve with mismatch penalty + optional hard cap
                           
                            result = simplify_proposition_mip(
                                X=x_ref_nonzero,
                                original_a=nonzero_original_a,
                                original_t=original_t,
                                coeff_candidate_sets=coeff_candidate_sets,
                                threshold_candidate_set=threshold_candidate_set,
                                cost_fn=default_complexity_cost,
                                margin=1e-16,
                                max_mismatches=int(delta*selected_samples_by_original_prop_j),         # exact matching on reference set
                                mismatch_penalty=1e-6 if delta > 0 else None,    # not needed if max_mismatches=0
                                verbose=False,
                            )

                            if result.mismatches > int(delta*selected_samples_by_original_prop_j):
                                original_a = c.w_[:,j]
                                original_t = -c.t_[j]

                                coeff_candidate_sets = [
                                    build_candidate_values(nonzero_original_a[j], sig_levels, include_original=True, use_round=True, use_trunc=True, use_integr=False)
                                    for j in range(len(nonzero_original_a))
                                ]
                                threshold_candidate_set = build_candidate_values(original_t, sig_levels, include_original=True, use_round=True, use_trunc=True, use_integr=False)

                                # Solve with mismatch penalty + optional hard cap
                                result = simplify_proposition_mip(
                                    X=x_ref_nonzero,
                                    original_a=nonzero_original_a,
                                    original_t=original_t,
                                    coeff_candidate_sets=coeff_candidate_sets,
                                    threshold_candidate_set=threshold_candidate_set,
                                    cost_fn=default_complexity_cost,
                                    margin=1e-16,
                                    max_mismatches=int(delta*selected_samples_by_original_prop_j),         # exact matching on reference set
                                    mismatch_penalty=1e-6 if delta > 0 else None,    # not needed if max_mismatches=0
                                    verbose=False,
                                )

                            simplified_a = np.zeros_like(original_a)
                            simplified_a[np.nonzero(original_a)[0]] = result.simplified_a
                            simple_model_by_mip_copy.conditions_[ci].w_[:,j] = simplified_a
                            simple_model_by_mip_copy.conditions_[ci].t_[j] = float(-result.simplified_t)
                    end_time = time.time()
                    model_df.loc[m,'comp_time'] = end_time - start_time
                    model_df.loc[m,'l2_lam'] = lam_reg
                    model_df.loc[m,'rep'] = ind
                    model_df.loc[m,'num_rules'] = nr+1
                    model_df.loc[m,'complexity'] = simple_model_by_mip_copy.model_complexity()
                    model_df.loc[m,'num_conditions'] = simple_model_by_mip_copy.num_of_rules()
                    model_df.loc[m,'num_propositions'] = simple_model_by_mip_copy.num_of_propositions()
                    model_df.loc[m,'train_loss'] = loss_fnc(y_train, simple_model_by_mip_copy.predict_aux(x_train)) if method == 'axis' else loss_fnc(y_train, simple_model_by_mip_copy.predict(x_train))
                    model_df.loc[m,'test_loss'] = loss_fnc(y_test, simple_model_by_mip_copy.predict_aux(x_test)) if method == 'axis' else loss_fnc(y_test, simple_model_by_mip_copy.predict(x_test))
                    model_df.loc[m,'ref_loss'] = loss_fnc(y_ref, simple_model_by_mip_copy.predict_aux(x_ref)) if method == 'axis' else loss_fnc(y_ref, simple_model_by_mip_copy.predict(x_ref))
                    if task == 'classification':
                        model_df.loc[m,'train_01_loss'] = zero_one_loss(y_train, np.where(simple_model_by_mip_copy.predict_aux(x_train) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_train, np.where(simple_model_by_mip_copy.predict(x_train) > 0.5, 1, 0))
                        model_df.loc[m,'test_01_loss'] = zero_one_loss(y_test, np.where(simple_model_by_mip_copy.predict_aux(x_test) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_test, np.where(simple_model_by_mip_copy.predict(x_test) > 0.5, 1, 0))
                        model_df.loc[m,'ref_01_loss'] = zero_one_loss(y_ref, np.where(simple_model_by_mip_copy.predict_aux(x_ref) > 0.5, 1, 0)) if method == 'axis' else zero_one_loss(y_ref, np.where(simple_model_by_mip_copy.predict(x_ref) > 0.5, 1, 0))
                    else:
                        model_df.loc[m,'train_01_loss'] = 0
                        model_df.loc[m,'test_01_loss'] = 0
                        model_df.loc[m,'ref_01_loss'] = 0
                    model_df.loc[m,'digits_complexity'] = simple_model_by_mip_copy.digit_complexity()
                    model_df.loc[m,'total_complexity'] = simple_model_by_mip_copy.model_complexity() + simple_model_by_mip_copy.digit_complexity()
                    model_df.loc[m,'digits'] = 'variable'
                    model_df.loc[m,'simplifying_method'] = 'mip'
                    model_df.loc[m,'seed'] = None
                    model_df.loc[m,'delta'] = delta
                    m += 1
                
                
                

    folder = './integer experiments/'
    model_df.to_excel(folder + 'canonical_baseline/simple_model_oblique_'+str(dataset_name)+'_including_integer_and_mismatches.xlsx', index=False)

In [48]:
dataset_name_list = [ 'diabetes', 'iris', 'liver', 'red_wine', 'breast_cancer', 'titanic', 'magic04', 'california_housing_price', 'used_car_price', 'make_friedman1', 'make_friedman2',
                     'make_friedman3', 'protein', 'parkinsons', 'adult', 'nhanes_diabetes', 'banknote', 'voice']
dataset_name_list = [ 'parkinsons', 'california_housing_price', 'nhanes_diabetes']
for dataset_name in dataset_name_list:
    run_experiment(dataset_name, method='oblique', mismaches_threshold = [0.0, 0.02, 0.06])
    print(f"Completed oblique experiment for {dataset_name}")


Completed oblique experiment for parkinsons
Completed oblique experiment for california_housing_price
Completed oblique experiment for nhanes_diabetes


In [82]:
import numpy as np
from decimal import Decimal

def count_decimal_digits(arr):
    def count(x):
        d = Decimal(str(x)).normalize()
        return max(0, -d.as_tuple().exponent)

    return np.array([count(x) for x in arr])


exm_arr = np.array([1.2340, 5.6789, 0.00123, -0.0])
count_decimal_digits(exm_arr)

array([3, 4, 5, 0])

In [128]:
lltb_original = lltb.LinRepRuleEnsemble(num_rules=14, task=dataset.learning_type, max_complexity=5, reg=10,
                                       epsilon_r=0.0, weighted_logloss=True, max_digits=16,
                                       no_refit=False, greedy=False, x_std=None, x_mean=None, axis_paral=False).fit(x_train,y_train)
lltb_original.rules(rounding_digits=14)

['-0.87015495837231',
 '0.03492938183498 if 2.99866083838294 MedInc +0.23267952703445 HouseAge -4.94203213436279 Latitude -4.44748581975592 Longitude ≥ 0.43549877238923',
 '0.24299938153485 if 1.57834572203471 MedInc +0.19905169660980 HouseAge -1.61533514714943 AveRooms -0.29519877697514 Longitude ≥ 0.30849761579032',
 '0.21852955254077 if MedInc ≥ 0.27123611310040',
 '0.28393219871444 if 0.93068971506789 MedInc +0.13998343859891 HouseAge -1.34061865599389 AveRooms -0.23932026702755 Latitude ≥ 0.15090374994164',
 '0.38072111318241 if MedInc ≥ 0.73314844244446 &  1.88593141591605 MedInc +0.71831692536166 HouseAge ≥ 2.44799788315395',
 '0.04583042647524 if 0.62026944430403 MedInc -0.33100678757667 Longitude ≥ 0.02807521367141',
 '0.82253193544609 if 0.15879220726831 MedInc -18.99603300368052 AveOccup -2.35114052659377 Latitude -1.92786634952334 Longitude ≥ 1.15473281852309',
 '0.18839429026430 if 0.57929933352633 MedInc +0.46554491474675 Population ≥ -0.08146962485240 &  0.98227268584609

In [129]:
lltb_simple_sequential = copy.deepcopy(lltb_original)
lltb_simple_sequential = lltb.SimplifiedRuleEnsemble(lltb_simple_sequential)
unique_train_idx = np.unique(train_ind)
x_ref = X.iloc[unique_train_idx].to_numpy()
y_ref = y[unique_train_idx]
lltb_simple_sequential, all_sequential_enumerations = lltb_simple_sequential.truncate_decimal_digits_param_by_param_extensively(x=x_ref, y=y_ref, return_all_enumerations = True,
                                                                                                   risk_increase_thrsh=0.0, round=True, seed=111, number_of_permutations=None)

sequential_decimal_digits_variation = []
for c_i, c in enumerate(lltb_simple_sequential.conditions_):
    nonzero_w_ind = [np.nonzero(lltb_original.conditions_[c_i].w_[:,j])[0] for j in range(lltb_original.conditions_[c_i].w_.shape[1])]
    sequential_decimal_digits_variation += list(count_decimal_digits(np.concatenate([c.w_[ind,j] for j, ind in enumerate(nonzero_w_ind)]))) + list(count_decimal_digits(c.t_))
sequential_decimal_digit_counts = np.sum(sequential_decimal_digits_variation)
lltb_simple_sequential.rules(rounding_digits=14), sequential_decimal_digit_counts


(['-0.87015495837231',
  '0.03492938183498 if 3.00000000000000 MedInc +0.23300000000000 HouseAge -4.94200000000000 Latitude -4.44750000000000 Longitude ≥ 0.43550000000000',
  '0.24299938153485 if 1.60000000000000 MedInc +0.20000000000000 HouseAge -1.60000000000000 AveRooms -0.30000000000000 Longitude ≥ 0.31000000000000',
  '0.21852955254077 if MedInc ≥ 0.27000000000000',
  '0.28393219871444 if 0.93000000000000 MedInc +0.14000000000000 HouseAge -1.34000000000000 AveRooms -0.24000000000000 Latitude ≥ 0.15000000000000',
  '0.38072111318241 if MedInc ≥ 0.73300000000000 &  1.88600000000000 MedInc +0.72000000000000 HouseAge ≥ 2.45000000000000',
  '0.04583042647524 if 0.62000000000000 MedInc -0.33000000000000 Longitude ≥ 0.02800000000000',
  '0.82253193544609 if 0.16000000000000 MedInc -19.00000000000000 AveOccup -2.35000000000000 Latitude -1.93000000000000 Longitude ≥ 1.16000000000000',
  '0.18839429026430 if 0.60000000000000 MedInc +0.47000000000000 Population ≥ -0.08000000000000 &  MedInc 